# RLHF 실습
> 이 실험 매뉴얼은 다음 자료를 번역·통합하였습니다: [블로그](https://newfacade.github.io/notes-on-reinforcement-learning/17-ppo-trl.html) & [trl 예제](https://github.com/huggingface/trl/blob/main/examples/notebooks/gpt2-sentiment.ipynb)

재현 실험 환경: 단일 GPU NVIDIA A800-SXM4-80GB, VRAM 사용량 10097MiB, 학습 소요 시간 35분 19초.

## PPO 동작 방식
1. **Rollout (롤아웃)**: 언어 모델이 query에 대한 응답(response)을 생성합니다.
2. **Evaluation (평가)**: 쿼리와 응답을 함수, 모델, 인간 피드백 또는 이들의 조합으로 평가합니다. 이 과정에서 각 쿼리/응답 쌍에 대해 **스칼라 값** 하나를 생성해야 합니다.
3. **Optimization (최적화)**: 최적화 단계에서는 쿼리/응답 쌍을 사용하여 시퀀스 내 토큰들의 로그 확률을 계산합니다. 이는 학습 중인 모델과 참조 모델(reference model) 두 가지로 수행됩니다. 두 출력 간의 KL 발산(KL-divergence)은 추가적인 보상 신호로 사용되어, 생성된 응답이 참조 언어 모델에서 너무 멀어지지 않도록 합니다. 이후 PPO를 사용하여 주 언어 모델을 학습합니다.
<div style="text-align: center">
<img src='figs/trl1.png' width='600'>
<p style="text-align: center;"> <b>그림:</b> PPO 흐름도 </p>
</div>

# 긍정적인 리뷰를 생성하도록 GPT-2 파인튜닝하기  
> BERT 감성 분류기를 보상 함수로 사용하여, GPT-2가 IMDB 영화 리뷰에서 긍정적인 내용을 생성하도록 최적화합니다.

<div style="text-align: center">
<img src='figs/gpt2_bert_training.png' width='600'>
<p style="text-align: center;"> <b>그림:</b> GPT-2 파인튜닝 실험 설정</p>
</div>

GPT-2를 IMDB 데이터셋 기반으로 긍정적인 영화 리뷰를 생성하도록 파인튜닝합니다. 모델은 실제 리뷰의 앞부분을 입력으로 받아 긍정적인 후속 내용을 생성해야 합니다. 긍정적인 후속 내용에 보상을 주기 위해, BERT 분류기를 사용하여 생성된 문장의 감성을 분석하고, 그 출력을 PPO 학습의 보상 신호로 활용합니다.

## 실험 설정

### 모델 및 데이터 다운로드
데이터셋
```bash
export HF_ENDPOINT=https://hf-mirror.com; huggingface-cli download --resume-download stanfordnlp/imdb --local-dir dataset/imdb --repo-type dataset
```
참조 모델
```bash
export HF_ENDPOINT=https://hf-mirror.com; huggingface-cli download --resume-download lvwerra/gpt2-imdb --local-dir model/gpt2-imdb
```
보상 모델
```bash
export HF_ENDPOINT=https://hf-mirror.com; huggingface-cli download --resume-download lvwerra/distilbert-imdb --local-dir model/distilbert-imdb
```

### 의존성 패키지 임포트

In [ ]:
# %pip install -r requirements.txt
# import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [2]:
import torch
from tqdm import tqdm
import pandas as pd

tqdm.pandas()

from transformers import pipeline, AutoTokenizer
from datasets import load_dataset

from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler

### 설정(Configuration)

In [3]:
config = PPOConfig(
    model_name="model/gpt2-imdb",
    learning_rate=1.41e-5,
    log_with="wandb",
)

sent_kwargs = {"top_k": None, "function_to_apply": "none", "batch_size": 16}

In [ ]:
import wandb

wandb.init()

`gpt2_imdb`라는 이름의 GPT-2 모델을 로드합니다. 이 모델은 Hugging Face의 [스크립트](https://github.com/huggingface/transformers/blob/main/examples/legacy/run_language_modeling.py)(특별한 설정 없음)를 사용하여 IMDB 데이터셋으로 1 에폭 추가 파인튜닝된 모델입니다. 나머지 파라미터들은 주로 원본 논문 《[Fine-Tuning Language Models from Human Preferences](https://huggingface.co/papers/1909.08593)》에서 가져왔습니다. 이 모델과 BERT 모델은 모두 Hugging Face 모델 허브([링크](https://huggingface.co/models))에서 확인할 수 있습니다.

## 데이터 및 모델 로드

### IMDB 데이터셋 로드  
IMDB 데이터셋은 50,000개의 영화 리뷰와 "긍정"/"부정" 감성 레이블로 구성되어 있습니다. 데이터셋을 DataFrame으로 로드한 뒤, 200자 이상의 리뷰만 필터링합니다. 이후 각 텍스트를 토크나이징하고, `LengthSampler`를 사용하여 지정된 길이로 무작위 잘라냅니다.

In [5]:
def build_dataset(
    config,
    dataset_name="dataset/imdb",
    input_min_text_length=2,
    input_max_text_length=8,
):
    """
    Build dataset for training. This builds the dataset from `load_dataset`, one should
    customize this function to train the model on its own dataset.

    Args:
        dataset_name (`str`):
            The name of the dataset to be loaded.

    Returns:
        dataloader (`torch.utils.data.DataLoader`):
            The dataloader for the dataset.
    """
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    tokenizer.pad_token = tokenizer.eos_token
    # load imdb with datasets
    ds = load_dataset(dataset_name, split="train")
    ds = ds.rename_columns({"text": "review"})
    ds = ds.filter(lambda x: len(x["review"]) > 200, batched=False)

    input_size = LengthSampler(input_min_text_length, input_max_text_length)

    def tokenize(sample):
        sample["input_ids"] = tokenizer.encode(sample["review"])[: input_size()]
        sample["query"] = tokenizer.decode(sample["input_ids"])
        return sample

    ds = ds.map(tokenize, batched=False)
    ds.set_format(type="torch")
    return ds

In [6]:
dataset = build_dataset(config)


def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

### 사전학습된 GPT-2 언어 모델 로드  
값 헤드(value head)를 포함한 GPT-2 모델과 토크나이저를 로드합니다. 모델은 두 번 로드합니다. 첫 번째 모델은 최적화(학습)에 사용하고, 두 번째 모델은 참조 모델로서 초기 체크포인트와의 KL 발산(KL-divergence)을 계산하는 데 사용합니다. 이는 PPO 학습에서 추가적인 보상 신호로 작용하여, 최적화된 모델이 원래 언어 모델에서 너무 멀어지지 않도록 합니다.

In [7]:
model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

tokenizer.pad_token = tokenizer.eos_token

### PPOTrainer 초기화  
`PPOTrainer`는 이후의 디바이스 할당 및 최적화를 담당합니다:

In [ ]:
ppo_trainer = PPOTrainer(
    config, model, ref_model, tokenizer, dataset=dataset, data_collator=collator
)

### BERT 분류기 로드  
IMDB 데이터셋으로 파인튜닝된 BERT 분류기를 로드합니다.

In [ ]:
device = ppo_trainer.accelerator.device
if ppo_trainer.accelerator.num_processes == 1:
    device = 0 if torch.cuda.is_available() else "cpu"  # to avoid a `pipeline` bug
sentiment_pipe = pipeline(
    "sentiment-analysis", model="model/distilbert-imdb", device=device
)

Device set to use cuda:0


모델은 부정(NEGATIVE) 클래스와 긍정(POSITIVE) 클래스의 logit 값을 출력합니다. 긍정 클래스의 logit 값을 언어 모델의 보상 신호로 사용합니다.

In [ ]:
text = "this movie was really bad!!"
sentiment_pipe(text, **sent_kwargs)

[{'label': 'NEGATIVE', 'score': 2.3350484371185303},
 {'label': 'POSITIVE', 'score': -2.726576089859009}]

In [11]:
text = "this movie was really good!!"
sentiment_pipe(text, **sent_kwargs)

[{'label': 'POSITIVE', 'score': 2.557040214538574},
 {'label': 'NEGATIVE', 'score': -2.294790267944336}]

### 생성 설정  
응답 생성 시 샘플링 방식만 사용하며, top-k 및 핵 샘플링(nucleus sampling)은 비활성화하고 최소 길이를 설정합니다.

In [12]:
gen_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}

## 모델 최적화

### 학습 루프

학습 루프는 다음 주요 단계들로 구성됩니다:
1. 정책 네트워크(GPT-2)로부터 쿼리에 대한 응답을 생성합니다.  
2. BERT를 사용하여 쿼리/응답 쌍의 감성을 측정합니다.  
3. (쿼리, 응답, 보상) 세 쌍을 이용하여 PPO로 정책을 최적화합니다.

In [13]:
output_min_length = 4
output_max_length = 16
output_length_sampler = LengthSampler(output_min_length, output_max_length)


generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}


for epoch, batch in enumerate(tqdm(ppo_trainer.dataloader)):
    query_tensors = batch["input_ids"]

    #### GPT-2로부터 응답 생성
    response_tensors = []
    for query in query_tensors:
        gen_len = output_length_sampler()
        generation_kwargs["max_new_tokens"] = gen_len
        query_response = ppo_trainer.generate(query, **generation_kwargs).squeeze()
        response_len = len(query_response) - len(query)
        response_tensors.append(query_response[-response_len:])
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    #### 감성 점수 계산
    texts = [q + r for q, r in zip(batch["query"], batch["response"])]
    pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
    positive_scores = [
        item["score"]
        for output in pipe_outputs
        for item in output
        if item["label"] == "POSITIVE"
    ]
    rewards = [torch.tensor(score) for score in positive_scores]

    #### PPO 스텝 실행
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
    ppo_trainer.log_stats(stats, batch, rewards)

100%|██████████| 194/194 [35:19<00:00, 10.92s/it]


### 학습 진행 상황  
Weights & Biases로 학습 진행 상황을 추적하면 아래 그림과 유사한 곡선을 볼 수 있습니다. wandb.ai의 인터랙티브 예제 리포트: [링크](https://wandb.ai/huggingface/trl/runs/w9l3110g).  
<div style="text-align: center">
<img src='figs/gpt2_tuning_progress.png' width='800'>
<p style="text-align: center;"> <b>그림:</b> 학습 중 평균 보상(reward)의 변화 추이 </p>
</div>  
몇 번의 최적화 단계를 거친 후, 모델이 더 긍정적인 출력을 생성하기 시작하는 것을 확인할 수 있습니다.

## 모델 검증  
IMDB 데이터셋에서 몇 가지 예제를 확인해 봅니다. `ref_model`을 사용하여 최적화된 모델 `model`과 최적화 전 참조 모델을 비교할 수 있습니다.

In [14]:
#### 데이터셋에서 배치 가져오기
bs = 16
game_data = dict()
dataset.set_format("pandas")
df_batch = dataset[:].sample(bs)
game_data["query"] = df_batch["query"].tolist()
query_tensors = df_batch["input_ids"].tolist()

response_tensors_ref, response_tensors = [], []

#### gpt2 및 gpt2_ref로부터 응답 생성
for i in range(bs):
    query = torch.tensor(query_tensors[i]).to(device)

    gen_len = output_length_sampler()
    query_response = ref_model.generate(
        query.unsqueeze(0), max_new_tokens=gen_len, **gen_kwargs
    ).squeeze()
    response_len = len(query_response) - len(query)
    response_tensors_ref.append(query_response[-response_len:])

    query_response = model.generate(
        query.unsqueeze(0), max_new_tokens=gen_len, **gen_kwargs
    ).squeeze()
    response_len = len(query_response) - len(query)
    response_tensors.append(query_response[-response_len:])

#### 응답 디코딩
game_data["response (before)"] = [
    tokenizer.decode(response_tensors_ref[i]) for i in range(bs)
]
game_data["response (after)"] = [
    tokenizer.decode(response_tensors[i]) for i in range(bs)
]

#### 최적화 전/후 쿼리/응답 쌍의 감성 분석
texts = [q + r for q, r in zip(game_data["query"], game_data["response (before)"])]
pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
positive_scores = [
    item["score"]
    for output in pipe_outputs
    for item in output
    if item["label"] == "POSITIVE"
]
game_data["rewards (before)"] = positive_scores

texts = [q + r for q, r in zip(game_data["query"], game_data["response (after)"])]
pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
positive_scores = [
    item["score"]
    for output in pipe_outputs
    for item in output
    if item["label"] == "POSITIVE"
]
game_data["rewards (after)"] = positive_scores

# 결과를 데이터프레임에 저장
df_results = pd.DataFrame(game_data)
df_results

생성된 시퀀스의 보상 평균/중앙값을 비교하면 학습 전후 간에 눈에 띄는 차이가 있음을 확인할 수 있습니다.

In [15]:
print("mean:")
display(df_results[["rewards (before)", "rewards (after)"]].mean())
print()
print("median:")
display(df_results[["rewards (before)", "rewards (after)"]].median())

mean:


rewards (before)    0.591666
rewards (after)     2.243934
dtype: float64


median:


rewards (before)    0.646912
rewards (after)     2.492161
dtype: float64

## 모델 저장  
마지막으로, 이후 사용을 위해 모델을 저장합니다.

In [16]:
model.save_pretrained("model/gpt2-imdb-pos-v2")
tokenizer.save_pretrained("model/gpt2-imdb-pos-v2")

('model/gpt2-imdb-pos-v2/tokenizer_config.json',
 'model/gpt2-imdb-pos-v2/special_tokens_map.json',
 'model/gpt2-imdb-pos-v2/vocab.json',
 'model/gpt2-imdb-pos-v2/merges.txt',
 'model/gpt2-imdb-pos-v2/added_tokens.json',
 'model/gpt2-imdb-pos-v2/tokenizer.json')